# Deepfake Group 31 - Model Training
This notebook defines an EfficientNet-B0 based detector and trains it with mixed precision and experiment tracking.

In [ ]:
from google.colab import drive
import os, random, numpy as np, torch

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Deepfake_Group31'
SEED = 42
os.makedirs(BASE_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"BASE_DIR set to: {BASE_DIR}")

## Install required dependencies
Install training and tracking dependencies in Colab.

In [ ]:
!pip install -q torch torchvision facenet-pytorch opencv-python scikit-learn matplotlib tqdm wandb

## Import training dependencies
Load PyTorch, torchvision, plotting, and wandb modules used in model training.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms, models
from torch.cuda.amp import autocast, GradScaler
import wandb

## Build dataset and loaders
Create a reusable dataset class and prepare 80/10/10 splits with `seed=42` and `batch_size=32`.

In [ ]:
class DeepfakeImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples = []
        class_map = {'real': 0, 'fake': 1}
        for class_name, label in class_map.items():
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for file_name in sorted(os.listdir(class_dir)):
                if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
                    self.samples.append((os.path.join(class_dir, file_name), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor([label], dtype=torch.float32)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

full_dataset = DeepfakeImageDataset(os.path.join(BASE_DIR, 'data'), transform=transform)
if len(full_dataset) == 0:
    raise ValueError('Dataset not found. Ensure BASE_DIR/data/{real,fake} is populated.')

target_total = 10000
if len(full_dataset) >= target_total:
    g = torch.Generator().manual_seed(SEED)
    sampled_idx = torch.randperm(len(full_dataset), generator=g)[:target_total].tolist()
    working_dataset = Subset(full_dataset, sampled_idx)
    split_lengths = [8000, 1000, 1000]
else:
    n = len(full_dataset)
    train_len = int(0.8 * n)
    val_len = int(0.1 * n)
    split_lengths = [train_len, val_len, n - train_len - val_len]
    working_dataset = full_dataset

g = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(working_dataset, split_lengths, generator=g)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

## Define EfficientNet-B0 based detector
Freeze backbone blocks `features[0]` to `features[4]`, then attach the custom classification head.

In [ ]:
class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        for idx in range(5):
            for param in backbone.features[idx].parameters():
                param.requires_grad = False

        backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(1280, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )
        self.model = backbone

    def forward(self, x):
        return self.model(x)

## Configure optimizer, scheduler, and logging
Set up BCELoss, Adam optimizer, StepLR scheduler, AMP scaler, and wandb run metadata.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepfakeDetector().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
scaler = GradScaler(enabled=torch.cuda.is_available())

os.makedirs(os.path.join(BASE_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, 'outputs'), exist_ok=True)
checkpoint_path = os.path.join(BASE_DIR, 'checkpoints', 'best_model.pth')
curves_path = os.path.join(BASE_DIR, 'outputs', 'training_curves.png')

wandb.init(
    project='deepfake_group31',
    config={
        'seed': SEED,
        'epochs': 15,
        'batch_size': 32,
        'learning_rate': 1e-4,
        'scheduler_step': 3,
        'scheduler_gamma': 0.5
    }
)

## Train for 15 epochs with mixed precision
Track `train_loss`, `val_loss`, and `val_accuracy` each epoch while saving the best checkpoint.

In [ ]:
num_epochs = 15
best_val_loss = float('inf')
train_losses, val_losses, val_accuracies = [], [], []

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_train_loss += loss.item() * images.size(0)

    epoch_train_loss = running_train_loss / len(train_loader.dataset)

    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * images.size(0)

            preds = (outputs >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.numel()

    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    epoch_val_acc = correct / total

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)

    scheduler.step()

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch + 1,
            'val_loss': epoch_val_loss
        }, checkpoint_path)

    wandb.log({
        'epoch': epoch + 1,
        'train_loss': epoch_train_loss,
        'val_loss': epoch_val_loss,
        'val_accuracy': epoch_val_acc,
        'lr': optimizer.param_groups[0]['lr']
    })

    print(f'Epoch {epoch+1:02d} | train_loss={epoch_train_loss:.4f} | val_loss={epoch_val_loss:.4f} | val_acc={epoch_val_acc:.4f}')

wandb.finish()

## Plot and save training curves
Generate loss and validation accuracy curves and save to `outputs/training_curves.png`.

In [ ]:
epochs = range(1, num_epochs + 1)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label='Train Loss')
plt.plot(epochs, val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, val_accuracies, label='Val Accuracy', color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.savefig(curves_path, dpi=200)
plt.show()
print(f'Training curves saved to: {curves_path}')